# Build PDFs locally (LaTeX + Typst)

**Prerequisites — run once as admin:**
```bash
# LaTeX
sudo apt-get install -y texlive-latex-base texlive-latex-recommended texlive-fonts-recommended \
  texlive-xetex texlive-latex-extra latexmk fonts-lmodern

# Node + MyST
sudo apt-get install -y nodejs npm
pip install mystmd

# Typst
pip install typst
```

In [ ]:
import yaml, copy, pathlib, subprocess, os

repo_root = pathlib.Path("..").resolve()
myst_path = repo_root / "myst.yml"
print(f"Repo root: {repo_root}")

## 1. Typst PDF

`myst.yml` already extends `export_typst.yml`.  
We only need to clone the template repo so Typst can find the bundled fonts.

In [ ]:
font_repo = repo_root / "typst_template"

if not font_repo.exists():
    print("Cloning typst_template for bundled fonts...")
    subprocess.run(
        ["git", "clone", "https://github.com/TUD-JB-OS/typst_template.git", str(font_repo)],
        check=True
    )

os.environ["TYPST_FONT_PATHS"] = str(font_repo / "src" / "assets" / "fonts")

subprocess.run(["myst", "build", "--typst"], cwd=repo_root, check=True)

## 2. LaTeX PDF

Temporarily swaps `myst.yml` extends to use `export_latex.yml`, builds, then restores.

In [ ]:
config = yaml.safe_load(myst_path.read_text())
original_extends = copy.deepcopy(config.get("extends", []))

try:
    config["extends"] = ["toc.yml", "export_latex.yml"]
    myst_path.write_text(yaml.dump(config, allow_unicode=True, sort_keys=False))
    print("myst.yml patched for LaTeX build")
    subprocess.run(["myst", "build", "--pdf"], cwd=repo_root, check=True)
finally:
    config["extends"] = original_extends
    myst_path.write_text(yaml.dump(config, allow_unicode=True, sort_keys=False))
    print("myst.yml restored")